# Analisis Tren Penjualan Coffee Shop — Kelompok 04

**Mata kuliah:** Analisis Data
**Materi:** Agregasi data tabular historis
**Dataset:** Coffee Shop Sales (Maven Roasters, Kaggle) — ±149 ribu transaksi dari 3 cabang di New York City, Januari–Juni 2023.

Notebook ini menjelaskan proses analisis langkah demi langkah: memuat data, membersihkan, membuat fitur baru,
lalu melakukan berbagai agregasi beserta visualisasinya. Semua fungsi pengolahan diambil dari `src/processing.py`
yang juga dipakai oleh dashboard Streamlit, sehingga angka di notebook dan dashboard selalu konsisten.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.express as px
from src import processing as P

pd.set_option("display.float_format", "{:,.2f}".format)
BLUE = "#2a78d6"
STORE_COLORS = {"Astoria": "#2a78d6", "Hell's Kitchen": "#eb6834", "Lower Manhattan": "#1baf7a"}

2026-10-28 12:42:20.028 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


## 1. Memuat data
Fungsi `load_data()` membaca `data/processed/coffee_sales.parquet` jika ada (jauh lebih cepat: ±1,5 detik dibanding ±9,5 detik untuk Excel), dan otomatis kembali ke file `.xlsx`/`.csv` jika parquet tidak ada. File parquet dibuat dengan `python -m src.processing --parquet` dan isinya identik dengan data Excel. Parsing dibuat aman untuk beberapa kemungkinan format:
- **Tanggal**: datetime asli Excel, angka serial Excel, atau teks `dd/mm/yyyy`.
- **Waktu**: objek waktu, teks `HH:MM:SS`, atau pecahan hari Excel.
- **unit_price**: angka biasa atau teks yang mengandung simbol `$` dan pemisah ribuan.

In [2]:
raw = P.load_data()
print(f"Jumlah baris: {len(raw):,} | Jumlah kolom: {raw.shape[1]}")
raw.head()

Jumlah baris: 149,116 | Jumlah kolom: 11


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,0 days 07:06:11,2,5,Lower Manhattan,32,3.00,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,0 days 07:08:56,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,0 days 07:14:04,2,5,Lower Manhattan,59,4.50,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,0 days 07:20:24,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,0 days 07:22:41,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [3]:
raw.dtypes

transaction_id                int64
transaction_date     datetime64[us]
transaction_time    timedelta64[us]
transaction_qty               int64
store_id                      int64
store_location                  str
product_id                    int64
unit_price                  float64
product_category                str
product_type                    str
product_detail                  str
dtype: object

## 2. Pemeriksaan kualitas data (cleaning)
Kita periksa missing value, baris duplikat, dan rentang nilai untuk mendeteksi anomali.

In [4]:
print("Missing value per kolom:")
print(raw.isna().sum())
print("\nBaris duplikat :", raw.duplicated().sum())
print("transaction_id duplikat :", raw["transaction_id"].duplicated().sum())
print("Rentang tanggal :", raw["transaction_date"].min().date(), "s/d", raw["transaction_date"].max().date())

Missing value per kolom:
transaction_id      0
transaction_date    0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64

Baris duplikat : 0
transaction_id duplikat : 0
Rentang tanggal : 2023-01-01 s/d 2023-06-30


In [5]:
raw[["transaction_qty", "unit_price"]].describe()

,transaction_qty,unit_price
count,"149,116.00","149,116.00"
mean,1.44,3.38
std,0.54,2.66
min,1.00,0.80
25%,1.00,2.50
50%,1.00,3.00
75%,2.00,3.75
max,8.00,45.00


**Temuan:** tidak ada missing value maupun duplikat. `transaction_id` unik per baris, sehingga satu baris dihitung sebagai satu transaksi.
Harga tertinggi ($45) berasal dari produk kemasan seperti biji kopi, jadi bukan kesalahan input. Data siap dipakai tanpa penghapusan baris.

In [6]:
raw.loc[raw["unit_price"] >= 20, ["product_category", "product_detail", "unit_price"]].drop_duplicates().head()

,product_category,product_detail,unit_price
3293,Coffee beans,Ethiopia,21.00
3296,Coffee beans,Organic Decaf Blend,28.00
3554,Coffee beans,Primo Espresso Roast,20.45
3916,Coffee beans,Organic Decaf Blend,22.50
4598,Branded,I Need My Bean! T-shirt,28.00


## 3. Feature engineering
Fungsi `add_features()` menambahkan:
- `revenue` = `transaction_qty × unit_price`
- `tanggal`, `bulan` (angka) dan `nama_bulan`
- `minggu` (tanggal Senin awal minggu)
- `hari` (nama hari, berurutan Senin–Minggu)
- `jam` (0–23)

In [7]:
df = P.add_features(raw)
df[["transaction_date", "transaction_time", "transaction_qty", "unit_price", "revenue",
    "nama_bulan", "minggu", "hari", "jam"]].head()

,transaction_date,transaction_time,transaction_qty,unit_price,revenue,nama_bulan,minggu,hari,jam
0,2023-01-01,0 days 07:06:11,2,3.00,6.00,Januari,2022-12-26,Minggu,7
1,2023-01-01,0 days 07:08:56,2,3.10,6.20,Januari,2022-12-26,Minggu,7
2,2023-01-01,0 days 07:14:04,2,4.50,9.00,Januari,2022-12-26,Minggu,7
3,2023-01-01,0 days 07:20:24,1,2.00,2.00,Januari,2022-12-26,Minggu,7
4,2023-01-01,0 days 07:22:41,2,3.10,6.20,Januari,2022-12-26,Minggu,7


**Verifikasi:** total revenue hasil agregasi harus sama dengan `sum(qty × unit_price)` pada data mentah.

In [8]:
total_raw = (raw["transaction_qty"] * raw["unit_price"]).sum()
total_agg = P.per_bulan(df)["revenue"].sum()
assert abs(total_raw - total_agg) < 1e-6
print(f"Total revenue mentah : ${total_raw:,.2f}")
print(f"Total revenue agregasi: ${total_agg:,.2f}  ✔ sama")

Total revenue mentah : $698,812.33
Total revenue agregasi: $698,812.33  ✔ sama


## 4. Ringkasan KPI

In [ ]:
bulanan = P.per_bulan(df)
kpi = pd.Series({
    "Total revenue": f"${df['revenue'].sum():,.2f}",
    "Jumlah transaksi": f"{len(df):,}",
    "Rata-rata per transaksi": f"${P.avg_transaction_value(df):,.2f}",
    "Total item terjual": f"{df['transaction_qty'].sum():,}",
    "MoM bulan terakhir": f"{bulanan['mom_pct'].iloc[-1]:+.1f}%",
})
kpi.to_frame("Nilai")

,Nilai
Total revenue,"$698,812.33"
Jumlah transaksi,"149,116"
Rata-rata per transaksi,$4.69
Total item terjual,"214,470"
MoM bulan terakhir,+6.2%


## 5. Agregasi waktu: harian & moving average 7 hari
Revenue harian berfluktuasi, sehingga kita pakai **moving average 7 hari** untuk menghaluskan pola mingguan dan memperlihatkan tren.

In [10]:
harian = P.per_hari(df)
fig = px.line(harian, x="tanggal", y="revenue", title="Revenue Harian & Moving Average 7 Hari",
              color_discrete_sequence=[BLUE], labels={"tanggal": "Tanggal", "revenue": "Revenue ($)"})
fig.update_traces(opacity=0.45, line_width=1, name="Harian", showlegend=True)
fig.add_scatter(x=harian["tanggal"], y=harian["ma7"], name="MA 7 hari", line=dict(color=BLUE, width=2.5))
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()
harian.describe()[["revenue", "transaksi"]]

,revenue,transaksi
count,181.00,181.00
mean,"3,860.84",823.85
min,"2,037.10",459.00
25%,"2,853.15",597.00
50%,"3,523.26",753.00
75%,"4,850.06","1,062.00"
max,"6,403.91","1,343.00"
std,"1,187.82",246.64


**Interpretasi:** tren revenue naik cukup konsisten dari Januari ke Juni. Penurunan di bulan Februari terutama karena jumlah harinya lebih sedikit dan masih musim dingin.

## 6. Agregasi mingguan

In [11]:
mingguan = P.per_minggu(df)
fig = px.line(mingguan, x="minggu", y="revenue", markers=True, title="Revenue Mingguan",
              color_discrete_sequence=[BLUE], labels={"minggu": "Minggu mulai", "revenue": "Revenue ($)"})
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

*Catatan:* minggu pertama dan terakhir tidak penuh 7 hari, sehingga nilainya terlihat lebih rendah.

## 7. Agregasi bulanan & pertumbuhan MoM
Pertumbuhan bulan ke bulan dihitung dengan rumus `(revenue bulan ini − revenue bulan lalu) / revenue bulan lalu × 100%`.

In [12]:
bulanan

,bulan,nama_bulan,revenue,transaksi,item,mom_pct
0,1,Januari,"81,677.74",17314,24870,NaN
1,2,Februari,"76,145.19",16359,23550,-6.77
2,3,Maret,"98,834.68",21229,30406,29.80
3,4,April,"118,941.08",25335,36469,20.34
4,5,Mei,"156,727.76",33527,48233,31.77
5,6,Juni,"166,485.88",35352,50942,6.23


In [13]:
lbl = [f"${r:,.0f}" + ("" if pd.isna(m) else f"<br>{m:+.1f}%") for r, m in zip(bulanan["revenue"], bulanan["mom_pct"])]
fig = px.bar(bulanan, x="nama_bulan", y="revenue", text=lbl, title="Revenue per Bulan & MoM %",
             color_discrete_sequence=[BLUE], labels={"nama_bulan": "Bulan", "revenue": "Revenue ($)"})
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

In [14]:
best = bulanan.loc[bulanan["revenue"].idxmax()]
print(f"Bulan tertinggi: {best['nama_bulan']} (${best['revenue']:,.2f})")
print(f"Pertumbuhan Jan→Jun: {(bulanan['revenue'].iloc[-1] / bulanan['revenue'].iloc[0] - 1) * 100:+.1f}%")

Bulan tertinggi: Juni ($166,485.88)
Pertumbuhan Jan→Jun: +103.8%


**Interpretasi:** setelah Februari turun, revenue melonjak di Maret–Mei. Juni menjadi bulan dengan revenue tertinggi.

## 8. Perbandingan cabang

In [15]:
cabang_bulan = P.per_cabang_bulan(df)
fig = px.line(cabang_bulan, x="nama_bulan", y="revenue", color="store_location", markers=True,
              color_discrete_map=STORE_COLORS, title="Tren Revenue Bulanan per Cabang",
              labels={"nama_bulan": "Bulan", "revenue": "Revenue ($)", "store_location": "Cabang"})
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

In [16]:
cabang = P.per_cabang(df)
cabang["avg_per_transaksi"] = cabang["revenue"] / cabang["transaksi"]
cabang["porsi_%"] = cabang["revenue"] / cabang["revenue"].sum() * 100
cabang

,store_location,revenue,transaksi,item,avg_per_transaksi,porsi_%
1,Hell's Kitchen,"236,511.17",50735,71737,4.66,33.84
0,Astoria,"232,243.91",50599,70991,4.59,33.23
2,Lower Manhattan,"230,057.25",47782,71742,4.81,32.92


**Interpretasi:** ketiga cabang punya revenue yang hampir seimbang (masing-masing sekitar sepertiga) dengan pola tren yang serupa. Hell's Kitchen sedikit unggul, sedangkan Lower Manhattan punya rata-rata nilai per transaksi tertinggi.

## 9. Pola waktu: heatmap hari × jam

In [17]:
heat = P.heatmap_hari_jam(df)
fig = px.imshow(heat, color_continuous_scale="Blues", aspect="auto", title="Heatmap Revenue Hari × Jam",
                labels=dict(x="Jam", y="Hari", color="Revenue ($)"))
fig.update_xaxes(dtick=1)
fig.show()

In [18]:
by_jam, by_hari = P.per_jam(df), P.per_hari_nama(df)
fig = px.bar(by_jam, x="jam", y="revenue", title="Revenue per Jam", color_discrete_sequence=[BLUE])
fig.update_xaxes(dtick=1); fig.update_yaxes(tickprefix="$", tickformat=",.0f"); fig.show()
fig = px.bar(by_hari, x="hari", y="revenue", title="Revenue per Hari", color_discrete_sequence=[BLUE])
fig.update_yaxes(tickprefix="$", tickformat=",.0f"); fig.show()
print(f"Jam tersibuk: {int(by_jam.loc[by_jam['revenue'].idxmax(), 'jam']):02d}:00")
print("Hari tertinggi:", by_hari.loc[by_hari['revenue'].idxmax(), 'hari'],
      "| terendah:", by_hari.loc[by_hari['revenue'].idxmin(), 'hari'])

Jam tersibuk: 10:00
Hari tertinggi: Senin | terendah: Sabtu


**Interpretasi:** puncak penjualan terjadi pada pagi hari (07:00–10:59), waktu orang berangkat kerja, lalu menurun setelah siang. Perbedaan antarhari relatif kecil.

## 10. Analisis produk

In [19]:
kategori = P.per_kategori(df)
kategori["porsi_%"] = kategori["revenue"] / kategori["revenue"].sum() * 100
fig = px.bar(kategori.sort_values("revenue"), x="revenue", y="product_category", orientation="h",
             title="Revenue per Kategori Produk", color_discrete_sequence=[BLUE])
fig.update_xaxes(tickprefix="$", tickformat=",.0f"); fig.show()
kategori

,product_category,revenue,transaksi,item,porsi_%
2,Coffee,"269,952.45",58416,89250,38.63
8,Tea,"196,405.95",45449,69737,28.11
0,Bakery,"82,315.64",22796,23214,11.78
4,Drinking Chocolate,"72,416.00",11468,17457,10.36
3,Coffee beans,"40,085.25",1753,1828,5.74
1,Branded,"13,607.00",747,776,1.95
6,Loose Tea,"11,213.60",1210,1210,1.60
5,Flavours,"8,408.80",6790,10511,1.20
7,Packaged Chocolate,"4,407.64",487,487,0.63


In [20]:
top10 = P.top_produk(df)
fig = px.bar(top10.sort_values("revenue"), x="revenue", y="product_detail", orientation="h",
             title="Top 10 Produk berdasarkan Revenue", color_discrete_sequence=[BLUE])
fig.update_xaxes(tickprefix="$", tickformat=",.0f"); fig.show()
top10

,product_detail,product_category,revenue,transaksi,item
75,Sustainably Grown Organic Lg,Drinking Chocolate,"21,151.75",2961,4453
20,Dark chocolate Lg,Drinking Chocolate,"21,006.00",3029,4668
48,Latte Rg,Coffee,"19,112.25",2896,4497
6,Cappuccino Lg,Coffee,"17,641.75",2772,4151
53,Morning Sunrise Chai Lg,Tea,"17,384.00",2830,4346
47,Latte,Coffee,"17,257.50",2990,4602
43,Jamaican Coffee River Lg,Coffee,"16,481.25",2911,4395
76,Sustainably Grown Organic Rg,Drinking Chocolate,"16,233.75",2842,4329
5,Cappuccino,Coffee,"15,997.50",2793,4266
2,Brazilian Lg,Coffee,"15,109.50",2771,4317


In [21]:
fig = px.treemap(P.per_kategori_tipe(df), path=[px.Constant("Semua"), "product_category", "product_type"],
                 values="revenue", color="revenue", color_continuous_scale="Blues",
                 title="Treemap Revenue: Kategori → Tipe Produk")
fig.show()

**Interpretasi:** Coffee dan Tea mendominasi revenue (sekitar dua pertiga dari total). Bakery berada di urutan ketiga. Produk berukuran besar (Lg) dan produk premium mendominasi daftar top 10.

## 11. Rata-rata nilai per transaksi (Average Transaction Value)

In [22]:
atv_bulan = bulanan.assign(atv=bulanan["revenue"] / bulanan["transaksi"])[["nama_bulan", "transaksi", "revenue", "atv"]]
print(f"ATV keseluruhan: ${P.avg_transaction_value(df):,.2f}")
atv_bulan

ATV keseluruhan: $4.69


,nama_bulan,transaksi,revenue,atv
0,Januari,17314,"81,677.74",4.72
1,Februari,16359,"76,145.19",4.65
2,Maret,21229,"98,834.68",4.66
3,April,25335,"118,941.08",4.69
4,Mei,33527,"156,727.76",4.67
5,Juni,35352,"166,485.88",4.71


**Interpretasi:** ATV relatif stabil di setiap bulan. Artinya, kenaikan revenue lebih banyak didorong oleh **bertambahnya jumlah transaksi**, bukan oleh naiknya nilai belanja per transaksi.

## 12. Kesimpulan
1. Revenue tumbuh kuat dari Januari ke Juni 2023, dengan Juni sebagai bulan tertinggi.
2. Ketiga cabang berkinerja hampir setara dengan tren yang sejalan.
3. Jam sibuk ada di pagi hari (07:00–10:59). Jadwal staf dan stok sebaiknya difokuskan pada jam ini.
4. Coffee & Tea adalah kategori utama. Bakery berpotensi dipaketkan (bundling) dengan minuman pagi.
5. Pertumbuhan didorong oleh volume transaksi, sedangkan nilai per transaksi relatif stabil.